# Cleaning4
Livello di pulizia che prende i file cleaned_01 fa le elaborazioni di cleaning 2 & 3 ma senza eliminare righe.\
File così creati servono per il merge.
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [19]:
file_codes = ['UCSDVOL']

#'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS'

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [20]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [21]:
len(zip_files)

1

## Operazioni
- Trasformare in dummies alcuni parametri
- Normalizzare i volumi
- nuovi metadati (cofattori e fattori)

In [29]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned4'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


The ADNI_variables_cleaned4 file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned4 file has restored the previous information of the file_code: ['UCSDVOL']
Open the file and verify it, if needed update the variables names and metadata


In [30]:
new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [ ]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')

    processed_df = df_new.copy(deep=True)

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in processed_df.columns]
    if to_dummy_list:
        print('\n\n Dummies in ----', file_name)
        processed_df, bool_var = dataCleaner.classes_to_dummies(processed_df, col_list=to_dummy_list) 
    
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n Volumes in ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(processed_df, file_code)
        print(processed_df.columns)
        # Transform volumes as ICV percentage
        processed_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    
    final_df = processed_df.copy(deep=True)
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_04')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_04', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
            new_support_file = infoSupportFile.get_subjects_and_multiplevisits(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 



 ---- UCSDVOL_28Oct2025_01.csv


 Volumes in ---- UCSDVOL_28Oct2025_01.csv
Index(['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'STATUS', 'Brain', 'ICV',
       'Ventricles', 'Hippocampus'],
      dtype='object')


In [32]:
final_df

,RID,VISCODE,VISIT_MONTH,EXAMDATE,STATUS,Brain%ICV,ICV%ICV,Ventricles%ICV,Hippocampus%ICV
0,2,sc,0,2005-08-26,complete,66.376611,100.0,6.495216,0.472903
1,3,sc,0,2005-09-01,complete,66.186512,100.0,5.107391,0.327410
2,3,m06,6,2006-03-13,complete,65.557551,100.0,5.331892,0.314240
3,3,m12,12,2006-09-12,complete,65.229907,100.0,5.488506,0.307518
4,3,m24,24,2007-09-12,complete,64.441220,100.0,6.029301,0.289132
...,...,...,...,...,...,...,...,...,...
2592,1425,m36,36,2010-08-16,complete,66.827458,100.0,2.003972,0.514626
2593,1427,sc,0,2007-08-20,complete,71.642369,100.0,1.494476,0.559996
2594,1430,sc,0,2007-09-07,complete,67.418097,100.0,2.258114,0.388360
2595,1430,m06,7,2008-04-04,complete,67.476466,100.0,2.324027,0.379025


In [33]:
updated_metadata

{'file_code': 'UCSDVOL',
 'level': 'cleaned_04',
 'population': ['ADNI1'],
 'source': 'ADNI',
 'cofattori': [],
 'predittori': ['Brain%ICV', 'ICV%ICV', 'Ventricles%ICV', 'Hippocampus%ICV'],
 'norm_scala': [],
 'norm_intervallo': [],
 'norm_volume': ['Brain%ICV', 'ICV%ICV', 'Ventricles%ICV', 'Hippocampus%ICV'],
 'norm_scale_value': [],
 'volume_norm_values': {'Brain%ICV': [60.16, 73.36, 'inverse'],
  'Ventricles%ICV': [0.11, 7.3, 'increasing'],
  'Hippocampus%ICV': [0.22, 0.66, 'inverse']}}